# M1 交互学习 Notebook

主教程是 `docs/M1_入门教程.md`。此 Notebook 可选，用于逐格执行相同的已实现步骤；默认终端路线不需要安装 Jupyter。

每个步骤在独立进程执行，始终使用项目 `.venv`。训练格会实际更新参数，但不会覆盖已有 Adapter。已有验证记录时可以跳过训练，直接验证重载。

In [ ]:
from pathlib import Path
import subprocess

ROOT = Path.cwd()
assert (ROOT / "scripts/m1.py").exists(), "请从 AI Waifu 项目目录打开 Notebook。"
PYTHON = ROOT / ".venv/bin/python"
assert PYTHON.exists(), "先按 README 准备 Python 环境。"

def run_step(step):
    subprocess.run([str(PYTHON), "-u", str(ROOT / "scripts/m1.py"), step], cwd=ROOT, check=True)


## 1 检查环境

MPS 是 PyTorch 使用苹果 GPU 的接口。这里真正执行一次张量计算，而不只检查设备标志。

In [ ]:
run_step("doctor")


## 2 下载基础模型

下载权重、Tokenizer、配置和聊天模板，并固定模型 commit。下载不是训练。

In [ ]:
run_step("download")


## 3 原模型推理

Prompt 改变这次输入，推理生成回答；此时没有更新参数。

In [ ]:
run_step("baseline")


## 4 检查监督标签

问题仍然被模型读取，但 loss 只覆盖目标回答。-100 是损失函数的忽略标记。

In [ ]:
run_step("inspect-data")


## 5 小样本 LoRA 训练

冻结基础权重 W，只更新新增的低秩参数 A、B。Trainer 提供训练循环，PEFT 提供 LoRA。8 次更新只验证流程。

In [ ]:
run_step("train")


## 6 新进程重新加载

新的 Python 进程重新加载基础模型与磁盘上的 Adapter，核对预测分数。回答文本相同不表示权重没变。

In [ ]:
run_step("verify")


## 7 汇总状态

全部真实记录通过且未过期才显示 M1 已验证；这不证明聊天质量提升。

In [ ]:
run_step("status")
